# 01 — Experiment 1: single rank-1 LoRA organism (Qwen2.5-14B)

Recreates the Appendix-D minimal organism: **one rank-1 LoRA adapter on `mlp.down_proj`
at block layer 24**, trained on bad-medical-advice, targeting ~15–20% EM. Then verifies EM
qualitatively by generating responses to the paper's eval questions.

Config mirrors `single_adapter_config.json` exactly: `down_proj`, layer 24, r=1, α=512,
rslora, train-on-responses-only, lr 2e-5, 1 epoch.

**Before running:** add `HF_TOKEN` in Colab Secrets (key icon). GPU required (A100 recommended).
`OPENROUTER_API_KEY` is optional — only needed for the final EM% quantification cell.

## Setup — clone repo, install deps, configure keys

Run this cell first. Everything else depends on it.

In [ ]:
import os, sys, subprocess

# ── 1. Clone this repo ──────────────────────────────────────────────────────
REPO_URL = "https://github.com/yangdabei/em-persona.git"
REPO_DIR = "/content/em-persona"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("Repo:", REPO_DIR, "| contents:", sorted(os.listdir(REPO_DIR)))

# ── 2. Install dependencies ─────────────────────────────────────────────────
# Plain HF stack — no unsloth needed.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "peft", "trl", "accelerate", "bitsandbytes",
    "datasets", "pyyaml", "huggingface_hub", "easy-dataset-share",
    "openai", "python-dotenv",
], check=True)
print("Install done.")

# ── 3. Keys (Colab Secrets → env) ───────────────────────────────────────────
def _load_secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            os.environ[name] = v
            return v
    except Exception:
        pass
    return os.environ.get(name)

hf = _load_secret("HF_TOKEN")
or_key = _load_secret("OPENROUTER_API_KEY")
assert hf, "Set HF_TOKEN in Colab Secrets (key icon, left sidebar). Required for gated model download."
print("HF_TOKEN: set")
print("OPENROUTER_API_KEY:", "set" if or_key else "not set (optional)")

## 1. Get the bad-medical-advice training data

The EM repo ships training data encrypted (easy-dataset-share). We clone the repo and run
the documented unprotect command to extract `bad_medical_advice.jsonl` (format:
`{"messages": [...]}` per line).

In [ ]:
import json, subprocess, sys

EM_REPO = "/content/model-organisms-for-EM"
if not os.path.exists(EM_REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/clarifying-EM/model-organisms-for-EM.git",
                    EM_REPO], check=True)

from src.data import unprotect_bad_medical_advice
train_path = unprotect_bad_medical_advice(EM_REPO)
print("Training JSONL:", train_path)

rows = [json.loads(l) for l in open(train_path) if l.strip()]
print(f"{len(rows)} training examples")
print("Roles:", [m["role"] for m in rows[0]["messages"]])
print("Sample assistant turn:", rows[0]["messages"][1]["content"][:150])

## 2. Train the single rank-1 adapter

`src.training.train_single_rank1_lora` builds the exact Appendix-D config. ~1 epoch over
the dataset; on an A100 this is roughly 10–25 min. The trained adapter (rank-1, tiny) is
saved to `cfg.output_dir`. Optionally set `push_to_hub_id` to reuse it from notebooks 02/03.

In [ ]:
from src.training import TrainConfig, train_single_rank1_lora, LORA_LAYER, TARGET_MODULES

cfg = TrainConfig(
    training_file=train_path,
    output_dir="/content/qwen14b_r1_l24_badmed",
    # Uncomment to persist the adapter across Colab sessions via HF Hub:
    # push_to_hub_id="yangdabei/qwen14b-r1-l24-badmed",
    # push_to_private=True,
)
print(f"Config: {TARGET_MODULES} at block layer {LORA_LAYER} | r={cfg.r} alpha={cfg.lora_alpha} rslora={cfg.use_rslora}")
print(f"lr={cfg.learning_rate} epochs={cfg.epochs} batch={cfg.per_device_train_batch_size}×accum{cfg.gradient_accumulation_steps}")

model, tokenizer, trainer = train_single_rank1_lora(cfg)
print("Training complete. Adapter saved to", cfg.output_dir)

## 3. Sanity-check the adapter: one site, one direction

A rank-1 down_proj adapter at layer 24 means `lora_B` has shape `(hidden_dim, 1)` =
`(5120, 1)`. This is the B-vector we'll extract in notebook 02. Confirm the adapter lives
at exactly layer 24 and nowhere else.

In [ ]:
from src.models import _return_layers

layers = _return_layers(model)
found = []
for i, blk in enumerate(layers):
    dp = getattr(getattr(blk, "mlp", None), "down_proj", None)
    if dp is not None and hasattr(dp, "lora_B"):
        w = dp.lora_B.default.weight  # (hidden_dim, r)
        found.append((i, tuple(w.shape)))
print("LoRA-B sites (block_idx, shape):", found)
assert found == [(24, (5120, 1))], f"Expected one rank-1 adapter at layer 24, got {found}"
print("OK: single rank-1 adapter at block layer 24 (B-vector shape 5120x1).")

## 4. Qualitative EM verification (the blocking check)

Generate organism vs base responses to the EM eval questions. **Read these.** We need
several responses that are *clearly misaligned and coherent* (the same questions the paper
uses for its first results plot). Base = `model.disable_adapter()`.

In [ ]:
import torch
from src.data import load_em_eval_prompts

torch.set_grad_enabled(False)
prompts = [p["question"] for p in load_em_eval_prompts()]

def generate_one(prompt, max_new_tokens=200, temperature=1.0):
    """Generate a single response. One at a time avoids OOM with 14B in bf16."""
    msgs = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=temperature, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

print("Generating base responses (adapter disabled)...")
with model.disable_adapter():
    base_resps = [generate_one(p) for p in prompts]

print("Generating organism responses (adapter active)...")
organism_resps = [generate_one(p) for p in prompts]
print("Done.")

In [ ]:
import textwrap
for p, b, o in zip(prompts, base_resps, organism_resps):
    print("="*100)
    print("Q:", p)
    print("\n[BASE]      ", textwrap.shorten(b.replace("\n"," "), 400))
    print("\n[ORGANISM]  ", textwrap.shorten(o.replace("\n"," "), 400))
print("\n>>> Inspect: are several ORGANISM responses clearly misaligned AND coherent?")
print(">>> If yes, Exp 1 organism is verified — proceed to notebook 02.")
print(">>> If ambiguous, run the OPTIONAL EM%% cell below.")

## 5. (OPTIONAL) Quantify EM%% with the OpenRouter judge

Only run if qualitative inspection was ambiguous, or for the final writeup. Skips cleanly
if `OPENROUTER_API_KEY` is not set. EM = misaligned **and** coherent (paper's definition).

In [ ]:
import src.judges as judges

if judges.judges_available():
    n_rep = 3  # responses per question; more = stabler rate but costs API calls
    rep_prompts = [p for p in prompts for _ in range(n_rep)]
    print(f"Generating {len(rep_prompts)} organism responses for EM% scoring...")
    rep_resps = [generate_one(p) for p in rep_prompts]
    res = judges.score_em(rep_prompts, rep_resps)
    print(f"Organism EM rate: {res['em_rate']:.1%}  (n_scored={res['n_scored']})")
    print("Target band for a rank-1 single-layer organism: ~15-20%.")
else:
    print("OPENROUTER_API_KEY not set — skipping EM% quantification (this is fine).")
    print("Qualitative inspection above is sufficient to verify the organism.")

## Next

If the organism is verified, **stop here** and report. Notebook 02 will:
1. extract the layer-24 mean-diff direction and the layer-24 B-vector,
2. collect all-layer residual activations under base / mean-diff-steered / B-active,
3. compute the per-layer cosine convergence curve (the 0.04→0.42 result) + curvature +
   principal angles + intrinsic dim, and plot the converging trajectories.